# NorCuts2
A free alternative to `opus.pro` and `vidyo.ai`

# Support:
[![](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)

# TODO📝
- [x] Release code
- [ ] Huggingface SpaceDemo
- [x] Two face in the cut
- [x] Custom caption and burn
- [x] Make the code faster
- [ ] More types of framing beyond 9:16

In [ ]:
#@title 🛠️ Installation
import os
import subprocess
import shutil
from IPython.display import clear_output

# 1. Full cleanup
print("🧹 Cleaning previous installation...")
%cd /content
if os.path.exists("NorCuts2"):
    shutil.rmtree("NorCuts2")

!git clone -b main https://github.com/zakiach555/NorCuts2.git
%cd /content/NorCuts2

# Pull latest code updates
!git pull

print("⏳ Installing UV package manager and system drivers...")

# 2. Install UV and Linux drivers
subprocess.run(['pip', 'install', 'uv'], check=True)
subprocess.run('sudo apt update -y && sudo apt install -y libcudnn8 xvfb', shell=True, check=True)
# Install static ffmpeg with NVENC (GPU encoder) support — apt version lacks NVENC
print("⏳ Installing ffmpeg with NVENC support...")
subprocess.run(
    "wget -q https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz"
    " -O /tmp/ffmpeg.tar.xz"
    " && tar -xf /tmp/ffmpeg.tar.xz -C /tmp/"
    " && cp /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg"
    " && cp /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe",
    shell=True, check=True
)

# 3. Create virtual environment
print("⏳ Creating virtual environment...")
subprocess.run(['uv', 'venv', '.venv'], check=True)

# 4. Install dependencies
print("⏳ Installing libraries...")

# Step A: WhisperX and basic requirements
cmds_fase_1 = [
    "uv pip install --python .venv git+https://github.com/m-bain/whisperx.git",
    "uv pip install --python .venv -r requirements.txt",
    "uv pip install --python .venv yt-dlp pytubefix",
    "uv pip install yt_dlp"
]

for cmd in cmds_fase_1:
    subprocess.run(cmd, shell=True, check=True)

# Step B: Alignment + Gemini fix
# - google-generativeai: required for Gemini
# - pandas: word segmentation
# - transformers==4.46.3: CRITICAL VERSION. Newer versions require Torch 2.6 and break alignment.
# - accelerate: speeds up model loading
print("🔨 Pinning Transformers to compatible version...")
extra_libs = [
    "uv pip install --python .venv google-generativeai",
    "uv pip install --python .venv pandas",
    "uv pip install --python .venv onnxruntime-gpu",
    "uv pip install --python .venv transformers==4.46.3 accelerate>=0.26.0"
]

for cmd in extra_libs:
    subprocess.run(cmd, shell=True, check=True)

# Step C: Final fix - stable Torch 2.3.1
# Reinstall last to ensure nothing silently upgraded it
print("🔨 Pinning stable Torch version (2.3.1)...")
cmd_fix_torch = (
    "uv pip install --python .venv "
    "torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 "
    "--index-url https://download.pytorch.org/whl/cu121"
)
subprocess.run(cmd_fix_torch, shell=True, check=True)

# Step D: Pin Numpy
print("🔨 Pinning Numpy...")
subprocess.run("uv pip install --python .venv 'numpy<2.0' setuptools==69.5.1", shell=True, check=True)

# Step E: Computer Vision fix (CRITICAL)
print("🔨 Fixing InsightFace and MediaPipe...")

# 1. Ensure InsightFace has the required engine
subprocess.run("uv pip install --python .venv insightface onnxruntime-gpu", shell=True, check=True)

# 2. Clean reinstall of MediaPipe
# Remove broken versions
subprocess.run("uv pip uninstall --python .venv mediapipe protobuf flatbuffers", shell=True)

# 3. Install flexible versions within critical bounds
# mediapipe>=0.10.0: latest version compatible with Py3.12
# protobuf>=3.20,<5.0: compatible without breaking legacy systems
subprocess.run("uv pip install --python .venv 'mediapipe>=0.10.0' 'protobuf>=3.20,<5.0' 'flatbuffers>=2.0'", shell=True, check=True)

# 5. Configure virtual display
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

clear_output()
print("✅ Installation complete!")
print("- Transformers 4.46.3 (alignment-compatible): INSTALLED")
print("- Torch 2.3.1: ACTIVE")

In [ ]:
#@title 🍪 YouTube Cookies (fix bot detection) - Optional
# Run this cell ONLY if you get "Sign in to confirm you are not a bot" errors.
#
# HOW TO EXPORT COOKIES (must be logged in to YouTube):
#  1. Install "Get cookies.txt LOCALLY" in Chrome/Edge:
#     https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc
#  2. Go to https://www.youtube.com (make sure you are signed in)
#  3. Click the extension icon -> Export -> Save as cookies.txt
#  4. Run this cell and upload the file

from google.colab import files
import subprocess, os

uploaded = files.upload()
for fname in uploaded:
    dest = "/content/NorCuts2/cookies.txt"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"Cookies saved to {dest} ({len(uploaded[fname])} bytes)")
    break

# Quick validation
print("
--- Validating cookies ---")
with open("/content/NorCuts2/cookies.txt", "r", errors="replace") as f:
    first_line = f.readline().strip()
    print(f"First line: {first_line[:80]}")
    if "Netscape" not in first_line and not first_line.startswith("#"):
        print("WARNING: File may not be in Netscape cookie format!")
    else:
        print("Format OK (Netscape)")

# Test cookies against a simple YouTube request
print("
--- Testing cookies against YouTube ---")
result = subprocess.run(
    ["/content/NorCuts2/.venv/bin/yt-dlp",
     "--cookies", "/content/NorCuts2/cookies.txt",
     "--skip-download", "--print", "title",
     "https://www.youtube.com/watch?v=uGRrWDi6AC4"],
    capture_output=True, text=True, timeout=30
)
if result.returncode == 0:
    print(f"✅ Cookies WORK! Video title: {result.stdout.strip()}")
else:
    print(f"❌ Cookies did NOT work.")
    print(result.stderr[-500:])
    print("
Try re-exporting fresh cookies from a browser actively logged in to YouTube.")


In [ ]:
#@title 🔑 API Configuration (Enter your keys here)
import json, os

# --- Enter your API keys below ---
# Leave blank to skip a provider. OpenRouter is recommended.
SELECTED_API    = "openrouter"    # Change to: openai / deepseek / zhipu / qwen / huggingface / gemini / g4f

OPENROUTER_KEY  = "YOUR_OPENROUTER_KEY_HERE"   # https://openrouter.ai
OPENAI_KEY      = "YOUR_OPENAI_KEY_HERE"   # https://platform.openai.com
DEEPSEEK_KEY    = "YOUR_DEEPSEEK_KEY_HERE"   # https://platform.deepseek.com
ZHIPU_KEY       = "YOUR_ZHIPU_KEY_HERE"   # https://open.bigmodel.cn
QWEN_KEY        = "YOUR_QWEN_KEY_HERE"   # https://dashscope.aliyuncs.com
HUGGINGFACE_KEY = "YOUR_HUGGINGFACE_KEY_HERE"   # https://huggingface.co/settings/tokens
GEMINI_KEY      = "YOUR_GEMINI_KEY_HERE"   # https://aistudio.google.com

# ---- Do not edit below this line ----
cfg = {
    "selected_api": SELECTED_API,
    "openrouter":   {"api_key": OPENROUTER_KEY,  "model": "deepseek/deepseek-v4-flash:free",              "chunk_size": 70000},
    "openai":       {"api_key": OPENAI_KEY,       "model": "gpt-4o-mini",                          "chunk_size": 70000},
    "deepseek":     {"api_key": DEEPSEEK_KEY,     "model": "deepseek-chat",                        "chunk_size": 60000},
    "zhipu":        {"api_key": ZHIPU_KEY,        "model": "glm-4-flash",                          "chunk_size": 50000},
    "qwen":         {"api_key": QWEN_KEY,         "model": "qwen-plus",                            "chunk_size": 60000},
    "huggingface":  {"api_key": HUGGINGFACE_KEY,  "model": "meta-llama/Llama-3.1-70B-Instruct",   "chunk_size": 20000},
    "gemini":       {"api_key": GEMINI_KEY,       "model": "gemini-2.5-flash",                     "chunk_size": 70000},
    "g4f":          {"model": "gpt-4o-mini",                                                        "chunk_size": 2000},
}

cfg_path = "/content/NorCuts2/api_config.json"
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=4)

print(f"Config saved -> selected_api: {SELECTED_API}")
active_key = cfg[SELECTED_API].get("api_key", "") if SELECTED_API != "g4f" else "(no key needed)"
print(f"Active key: {active_key[:8]}..." if active_key and active_key != "(no key needed)" else "WARNING: No API key set for", SELECTED_API)


In [ ]:
#@title 🚀 Setup & Run
import os
import glob

%cd /content/NorCuts2

# Pull latest code updates
!git pull

# Show repo structure so we know what was cloned
print("📁 Cloned repo contents:")
!find /content/NorCuts2 -maxdepth 3 -name "*.py" | head -30

# 1. Set up virtual display (avoids GUI errors)
os.system('Xvfb :1 -screen 0 2560x1440x8 &')
os.environ['DISPLAY'] = ':1.0'

# 2. Force headless Matplotlib backend:
# Forces Matplotlib to use 'Agg' backend (headless mode)
# This ignores the 'module://matplotlib_inline.backend_inline' that Colab forces and breaks the script.
os.environ['MPLBACKEND'] = 'Agg'

# Find app.py dynamically
matches = glob.glob('/content/NorCuts2/**/app.py', recursive=True)
if not matches:
    raise FileNotFoundError('app.py not found in the cloned repo. Make sure your code is pushed to https://github.com/zakiach555/NorCuts2.git')
app_path = matches[0]
print(f"✅ Found app at: {app_path}")

print("🚀 Starting NorCuts2 using the virtual environment...")
print("⚠️ Ignore 'UserWarning' messages, wait for the public URL link.")

# Run using the virtual environment Python
!"/content/NorCuts2/.venv/bin/python" "{app_path}" --colab


 .1v Alpha<br>

A free alternative to `opus.pro` and `vidyo.ai`<br>


`0.1v Alpha`<br>

Apenas A free alternative to `opus.pro` and `vidyo.ai`<br>
